# LLaMA 3 8B Instruct

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, Conversation, BitsAndBytesConfig, set_seed
from evaluate import load
import torch
import json
import os
import ast
import random
from vqa_evaluation_prompts import *
import pandas as pd
import numpy as np

/home/xuezheng/anaconda3/envs/llama3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
set_seed(20)
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
model = AutoModelForCausalLM.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quantization_config, device_map="auto", num_beams = 5)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Meta-Llama-3-8B-Instruct", quantization_config=quantization_config, device_map="auto")

Loading checkpoint shards: 100%|██████████| 4/4 [00:02<00:00,  1.45it/s]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [3]:
spearman_list = []
pearson_list = []

## Rule 1

In [4]:
with open("reasoning_evaluation_support_set.json", 'r') as file:
    rule1_data = json.load(file)['rule1']

In [5]:
final_mark_list = []
id_list = []

for instance in rule1_data:
    print(instance)
    id_list.append(instance['image_id'])
    chatbot = pipeline(task="conversational", model=model, tokenizer=tokenizer)

    conversation = Conversation([{"role": "system", "content": SYSTEM_PROMPT_RULE_1}])
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": FEW_SHOT_PROMPT})
    conversation = chatbot(conversation)

    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE1_PROMPT_0000545})
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE1_PROMPT_0000007})
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE1_PROMPT_0000019})
    conversation = chatbot(conversation)

    conversation.add_message({"role": "user", "content": f"Please evaluate the following reasoning:\n\n Candidate reasoning: {instance['candidate']}\n\n Reference reasoning{instance['reference']}"})
    conversation = chatbot(conversation)
    reply = conversation.messages[-1]["content"]
    print(reply)

    conversation.add_message({"role": "user", "content": USER_PROMPT_FINAL})
    conversation = chatbot(conversation)
    final_mark = conversation.messages[-1]["content"]
    final_mark_dict= ast.literal_eval(final_mark)
    print(final_mark_dict)

    final_mark_list.append(final_mark_dict)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


{'image_id': '0000005', 'model': 'GPT', 'candidate': 'The person on foot at the construction site is not wearing a hard hat, and the clothes do not cover the shoulders and legs completely. Shoes are not visible in the image.', 'reference': 'Person on the left not using PPE.', 'evaluation': 'Relevance: 2 mark. The candidate reasoning is highly relevant to the safety rule, as it revolves around the use of PPE. Equivalence: 2 mark. The candidate explanation mentions the same reason for violation. Specificity: 1 mark. The candidate explanation mentions the person on foot, but does not mention a specific location or attribute that makes him distinguishable from the image.', 'mark': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}}


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end gene

Let me evaluate the candidate explanation based on the three criteria: Relevance, Equivalence, and Specificity.

Relevance: 2 mark. The candidate reasoning is highly relevant to the safety rule, as it mentions the use of basic PPE (hard hat, clothes covering shoulders and legs) and the lack of it.

Equivalence: 2 mark. The candidate explanation mentions the person on foot at the construction site not using PPE, which is the same as the reference explanation. The reference explanation only mentions one person, but the candidate explanation provides more details about the PPE violations.

Specificity: 1 mark. The candidate explanation provides some specific information about the violator, such as the lack of shoes being visible in the image. However, it does not pinpoint the violator as the person on the left, which is mentioned in the reference explanation.

Mark: {"relevance": 2, "equivalence": 2, "specificity": 1, "total": 5}
{'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total

In [6]:
to_save = dict(zip(id_list, final_mark_list))

with open("temp1.json", 'w') as file:
    json.dump(to_save, file)

In [7]:
# Load evaluations
with open("temp1.json", 'r') as file:
    llama_evaluation = json.load(file)

human_evaluation = {}
for instance in rule1_data:
    human_evaluation.update({instance['image_id']: instance['mark']})

In [8]:
print(llama_evaluation)
print(human_evaluation)

{'0000005': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0000007': {'relevance': 2, 'equivalence': 0, 'specificity': 2, 'total': 4}, '0000019': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000039': {'relevance': 2, 'equivalence': 0, 'specificity': 2, 'total': 4}, '0000046': {'relevance': 1, 'equivalence': 0, 'specificity': 1, 'total': 2}, '0000786': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000956': {'relevance': 2, 'equivalence': 1, 'specificity': 1, 'total': 4}, '0000545': {'relevance': 0, 'equivalence': 0, 'specificity': 0, 'total': 0}, '0000912': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}}
{'0000005': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0000007': {'relevance': 2, 'equivalence': 1, 'specificity': 2, 'total': 5}, '0000019': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000039': {'relevance': 2, 'equivalence': 1, 'specificity': 2, 'total': 5

In [9]:
# Save evaluation marks to pandas dataframe
llama_eval_num = []
human_eval_num = []
for image_id, _ in human_evaluation.items():
    human_eval_num.append(float(human_evaluation[image_id]['total']))
    llama_eval_num.append(float(llama_evaluation[image_id]['total']))

print(sum(human_eval_num)/len(human_eval_num))
print(sum(llama_eval_num)/len(llama_eval_num))

df = pd.DataFrame({'human': np.array(human_eval_num), 'llama': np.array(llama_eval_num)})
pearson_corr = df.corr('pearson')
spearman_corr = df.corr('spearman')

print("Pearson result:")
print(pearson_corr)
print("Spearman result:")
print(spearman_corr)

4.444444444444445
4.111111111111111
Pearson result:
          human     llama
human  1.000000  0.905738
llama  0.905738  1.000000
Spearman result:
          human     llama
human  1.000000  0.928571
llama  0.928571  1.000000


In [10]:
spearman_list.append(spearman_corr['llama']['human'])
pearson_list.append(pearson_corr['llama']['human'])

## Rule 2

In [11]:
with open("reasoning_evaluation_support_set.json", 'r') as file:
    rule2_data = json.load(file)['rule2']

final_mark_list = []
id_list = []

for instance in rule2_data:
    print(instance)
    id_list.append(instance['image_id'])
    chatbot = pipeline(task="conversational", model=model, tokenizer=tokenizer)

    conversation = Conversation([{"role": "system", "content": SYSTEM_PROMPT_RULE_2}])
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": FEW_SHOT_PROMPT})
    conversation = chatbot(conversation)

    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE2_PROMPT_0000925})
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE2_PROMPT_0003632})
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE2_PROMPT_0004235})
    conversation = chatbot(conversation)

    conversation.add_message({"role": "user", "content": f"Please evaluate the following reasoning:\n\n Candidate reasoning: {instance['candidate']}\n\n Reference reasoning{instance['reference']}"})
    conversation = chatbot(conversation)
    reply = conversation.messages[-1]["content"]
    print(reply)

    conversation.add_message({"role": "user", "content": USER_PROMPT_FINAL})
    conversation = chatbot(conversation)
    final_mark = conversation.messages[-1]["content"]
    final_mark_dict= ast.literal_eval(final_mark)
    print(final_mark_dict)

    final_mark_list.append(final_mark_dict)

{'image_id': '0000925', 'model': 'GPT', 'candidate': 'A worker is at a height greater than three meters without a safety harness and the edges are without any edge protection.', 'reference': 'The worker in grey is not wearing safety harness when working at the edge of the roof.', 'evaluation': 'Relevance: 2 mark. The candidate reasoning is talking about safety harness. Equivalence: 2 mark. Both the candidate and the reference are talking about the same violation of not wearing harness. Specificity: 1 mark. The candidate only says the worker at height.', 'mark': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}}
I'll evaluate the candidate explanation based on the three criteria: Relevance, Equivalence, and Specificity.

* Relevance: 2 marks (the candidate explanation is directly related to the safety rule, mentioning safety harnesses and edge protection)
* Equivalence: 2 marks (both the candidate and the reference are talking about the same violation: not wearing a safet

In [12]:
to_save = dict(zip(id_list, final_mark_list))

with open("temp2.json", 'w') as file:
    json.dump(to_save, file)

In [13]:
# Load evaluations
with open("temp2.json", 'r') as file:
    llama_evaluation = json.load(file)

human_evaluation = {}
for instance in rule2_data:
    human_evaluation.update({instance['image_id']: instance['mark']})

In [14]:
print(llama_evaluation)
print(human_evaluation)

{'0000925': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000149': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0004235': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0003632': {'relevance': 0, 'equivalence': 0, 'specificity': 0, 'total': 0}, '0002815': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0003479': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0001530': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000328': {'relevance': 2, 'equivalence': 1, 'specificity': 1, 'total': 4}, '0000993': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0001699': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}}
{'0000925': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0000149': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0004235': {'relevance': 2, 'equivalence': 1, 'specificity': 0, 'total': 3

In [15]:
# Save evaluation marks to pandas dataframe
llama_eval_num = []
human_eval_num = []
for image_id, _ in human_evaluation.items():
    human_eval_num.append(float(human_evaluation[image_id]['total']))
    llama_eval_num.append(float(llama_evaluation[image_id]['total']))

print(sum(human_eval_num)/len(human_eval_num))
print(sum(llama_eval_num)/len(llama_eval_num))

df = pd.DataFrame({'human': np.array(human_eval_num), 'llama': np.array(llama_eval_num)})
pearson_corr = df.corr('pearson')
spearman_corr = df.corr('spearman')

print("Pearson result:")
print(pearson_corr)
print("Spearman result:")
print(spearman_corr)

4.4
4.8
Pearson result:
          human     llama
human  1.000000  0.855356
llama  0.855356  1.000000
Spearman result:
          human     llama
human  1.000000  0.505193
llama  0.505193  1.000000


In [16]:
spearman_list.append(spearman_corr['llama']['human'])
pearson_list.append(pearson_corr['llama']['human'])

## Rule 3

In [17]:
with open("reasoning_evaluation_support_set.json", 'r') as file:
    rule3_data = json.load(file)['rule3']

final_mark_list = []
id_list = []

for instance in rule3_data:
    print(instance)
    id_list.append(instance['image_id'])
    chatbot = pipeline(task="conversational", model=model, tokenizer=tokenizer)

    conversation = Conversation([{"role": "system", "content": SYSTEM_PROMPT_RULE_2}])
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": FEW_SHOT_PROMPT})
    conversation = chatbot(conversation)

    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE3_PROMPT_0001597})
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE3_PROMPT_0000007})
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE3_PROMPT_0000117})
    conversation = chatbot(conversation)

    conversation.add_message({"role": "user", "content": f"Please evaluate the following reasoning:\n\n Candidate reasoning: {instance['candidate']}\n\n Reference reasoning{instance['reference']}"})
    conversation = chatbot(conversation)
    reply = conversation.messages[-1]["content"]
    print(reply)

    conversation.add_message({"role": "user", "content": USER_PROMPT_FINAL})
    conversation = chatbot(conversation)
    final_mark = conversation.messages[-1]["content"]
    final_mark_dict= ast.literal_eval(final_mark)
    print(final_mark_dict)

    final_mark_list.append(final_mark_dict)

{'image_id': '0000007', 'model': 'GPT', 'candidate': 'There is no visible edge protection for the area where workers are standing which appears to be elevated.', 'reference': 'Opening not protected on both the left and the right of the images.', 'evaluation': 'Relevance: 2 mark. The candidate reasoning is talking about edge protection. Equivalence: 2 mark. They are both talking about absence of edge protection. Specificity: 1 mark. The candidate mentions the the place where a worker is standing.', 'mark': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}}
Here's my evaluation:

* Relevance: 2 marks because the candidate explanation is directly related to the safety rule about edge protection when working from a height of three meters.
* Equivalence: 2 marks because the candidate and reference explanations are discussing the same issue, which is the absence of edge protection.
* Specificity: 2 marks because the candidate explanation provides specific details about the loc

In [18]:
to_save = dict(zip(id_list, final_mark_list))

with open("temp3.json", 'w') as file:
    json.dump(to_save, file)

In [19]:
# Load evaluations
with open("temp3.json", 'r') as file:
    llama_evaluation = json.load(file)

human_evaluation = {}
for instance in rule3_data:
    human_evaluation.update({instance['image_id']: instance['mark']})

In [20]:
print(llama_evaluation)
print(human_evaluation)

{'0000007': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000117': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000328': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000468': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0001229': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0001274': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0001484': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0001597': {'relevance': 0, 'equivalence': 0, 'specificity': 1, 'total': 1}, '0002367': {'relevance': 0, 'equivalence': 0, 'specificity': 0, 'total': 0}, '0002925': {'relevance': 0, 'equivalence': 0, 'specificity': 1, 'total': 1}}
{'0000007': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0000117': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000328': {'relevance': 2, 'equivalence': 0, 'specificity': 2, 'total': 4

In [21]:
# Save evaluation marks to pandas dataframe
llama_eval_num = []
human_eval_num = []
for image_id, _ in human_evaluation.items():
    human_eval_num.append(float(human_evaluation[image_id]['total']))
    llama_eval_num.append(float(llama_evaluation[image_id]['total']))

print(sum(human_eval_num)/len(human_eval_num))
print(sum(llama_eval_num)/len(llama_eval_num))

df = pd.DataFrame({'human': np.array(human_eval_num), 'llama': np.array(llama_eval_num)})
pearson_corr = df.corr('pearson')
spearman_corr = df.corr('spearman')

print("Pearson result:")
print(pearson_corr)
print("Spearman result:")
print(spearman_corr)

3.7
4.2
Pearson result:
          human     llama
human  1.000000  0.922602
llama  0.922602  1.000000
Spearman result:
          human     llama
human  1.000000  0.638667
llama  0.638667  1.000000


In [22]:
spearman_list.append(spearman_corr['llama']['human'])
pearson_list.append(pearson_corr['llama']['human'])

## Rule 4

In [23]:
with open("reasoning_evaluation_support_set.json", 'r') as file:
    rule4_data = json.load(file)['rule4']

final_mark_list = []
id_list = []

for instance in rule4_data:
    print(instance)
    id_list.append(instance['image_id'])
    chatbot = pipeline(task="conversational", model=model, tokenizer=tokenizer)

    conversation = Conversation([{"role": "system", "content": SYSTEM_PROMPT_RULE_2}])
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": FEW_SHOT_PROMPT})
    conversation = chatbot(conversation)

    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE4_PROMPT_0001512})
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE4_PROMPT_0004725})
    conversation = chatbot(conversation)
    conversation.add_message({"role": "user", "content": EXAMPLE_EVAL_RULE4_PROMPT_0002093})
    conversation = chatbot(conversation)

    conversation.add_message({"role": "user", "content": f"Please evaluate the following reasoning:\n\n Candidate reasoning: {instance['candidate']}\n\n Reference reasoning{instance['reference']}"})
    conversation = chatbot(conversation)
    reply = conversation.messages[-1]["content"]
    print(reply)

    conversation.add_message({"role": "user", "content": USER_PROMPT_FINAL})
    conversation = chatbot(conversation)
    final_mark = conversation.messages[-1]["content"]
    final_mark_dict= ast.literal_eval(final_mark)
    print(final_mark_dict)

    final_mark_list.append(final_mark_dict)

{'image_id': '0000327', 'model': 'GPT', 'candidate': 'A worker is standing in the operation radius of an excavator without maintaining a safe distance, which could be dangerous if the excavator is in operation.', 'reference': 'The worker holding an umbrella is too close to the excavator is operation.', 'evaluation': 'Relevance: 2 mark. The candidate reasoning is about blind spot of excavator. Equivalence: 2 mark. They are both talking about worker too close to the excavator. Specificity: 1 mark. The candidate does not mention a specific position of the worker and any attributes of the worker.', 'mark': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}}
Here's my evaluation:

* Relevance: The candidate reasoning is highly relevant to the safety rule, as it's related to the use of safety harness when working from a height of three meters and the edges are without any edge protection, and it mentions the operation radius of the excavator. (Mark: 2)
* Equivalence: The candid

In [24]:
to_save = dict(zip(id_list, final_mark_list))

with open("temp4.json", 'w') as file:
    json.dump(to_save, file)

In [25]:
# Load evaluations
with open("temp4.json", 'r') as file:
    llama_evaluation = json.load(file)

human_evaluation = {}
for instance in rule4_data:
    human_evaluation.update({instance['image_id']: instance['mark']})

In [26]:
print(llama_evaluation)
print(human_evaluation)

{'0000327': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0000878': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0004725': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0001512': {'relevance': 1, 'equivalence': 0, 'specificity': 1, 'total': 2}, '0000386': {'relevance': 0, 'equivalence': 1, 'specificity': 0, 'total': 1}, '0001658': {'relevance': 2, 'equivalence': 0, 'specificity': 2, 'total': 4}, '0003420': {'relevance': 1, 'equivalence': 0, 'specificity': 0, 'total': 1}, '0001325': {'relevance': 2, 'equivalence': 2, 'specificity': 2, 'total': 6}, '0002093': {'relevance': 0, 'equivalence': 0, 'specificity': 0, 'total': 0}}
{'0000327': {'relevance': 1, 'equivalence': 2, 'specificity': 1, 'total': 4}, '0000878': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0004725': {'relevance': 2, 'equivalence': 2, 'specificity': 1, 'total': 5}, '0001512': {'relevance': 2, 'equivalence': 0, 'specificity': 0, 'total': 2

In [27]:
# Save evaluation marks to pandas dataframe
llama_eval_num = []
human_eval_num = []
for image_id, _ in human_evaluation.items():
    human_eval_num.append(float(human_evaluation[image_id]['total']))
    llama_eval_num.append(float(llama_evaluation[image_id]['total']))

print(sum(human_eval_num)/len(human_eval_num))
print(sum(llama_eval_num)/len(llama_eval_num))

df = pd.DataFrame({'human': np.array(human_eval_num), 'llama': np.array(llama_eval_num)})
pearson_corr = df.corr('pearson')
spearman_corr = df.corr('spearman')

print("Pearson result:")
print(pearson_corr)
print("Spearman result:")
print(spearman_corr)

2.888888888888889
3.5555555555555554
Pearson result:
          human     llama
human  1.000000  0.961034
llama  0.961034  1.000000
Spearman result:
          human     llama
human  1.000000  0.928485
llama  0.928485  1.000000


In [28]:
spearman_list.append(spearman_corr['llama']['human'])
pearson_list.append(pearson_corr['llama']['human'])

In [29]:
spearman_avg = sum(spearman_list)/len(spearman_list)
pearson_avg = sum(pearson_list)/len(pearson_list)
print(spearman_avg)
print(pearson_avg)

0.7502291465875028
0.9111824698039588
